# Kimi-K3 (GDN-2) Code LM — train on **Kaggle (2x T4)**

Trains the model on **CodeParrot** using `configs/kaggle_2xt4.yaml`. The training loop
auto-detects both T4s and runs **data-parallel** (params replicated, each batch sharded
across the 2 GPUs) — no extra flags needed.

**Before you start (Notebook sidebar):**
* `Settings -> Accelerator = GPU T4 x2`
* `Settings -> Internet = On`  (required to pip-install and stream CodeParrot)


## 0. Confirm both GPUs

In [ ]:
!nvidia-smi

In [ ]:
try:
    from huggingface_hub import notebook_login
    notebook_login()
except Exception as exc:
    print(f"Hugging Face login unavailable: {exc}")

## 1. Get the project

Set `REPO_URL` to your repo (Internet must be on). Alternatively, attach the project as
a Kaggle Dataset and point `PROJECT_DIR` at it. We work inside `/kaggle/working` so
outputs (checkpoints) persist with the notebook.


In [ ]:
import os

REPO_URL = "https://github.com/wisnunugroho21/nugie-kimi-k3.git"  # <-- EDIT ME
PROJECT_DIR = "/kaggle/working/nugie-kimi-k3"

if not os.path.isdir(PROJECT_DIR):
    !git clone $REPO_URL $PROJECT_DIR
%cd $PROJECT_DIR
!ls


In [ ]:
%cd /kaggle/working/nugie-kimi-k3
!git pull

## 2. Install dependencies

Requirements first, then the CUDA 12 JAX build last so the GPU plugin wins.


In [ ]:
!pip install -q -U -r requirements.txt
!pip install -q -U "jax[cuda12]"


## 3. Verify JAX sees **both** T4s

`jax.device_count()` must be 2 for data-parallel training. If it prints 1, set the
accelerator to *GPU T4 x2* and restart the session.


In [ ]:
import jax
print("JAX", jax.__version__, "devices:", jax.devices())
n = jax.device_count()
print("device_count:", n)
assert n == 2, "Expected 2 GPUs — set Accelerator = GPU T4 x2 and restart."


## 4. Build the config

Loads `configs/kaggle_2xt4.yaml` (batch 16 -> 8 per GPU) with demo-friendly overrides.
Data and checkpoints go under `/kaggle/working` so they survive as notebook output.
`batch_size` must stay divisible by 2 (the device count).


In [ ]:
from pipeline.config import ExperimentConfig

cfg = ExperimentConfig.load("configs/kaggle_2xt4.yaml")

cfg.data.data_dir   = "/kaggle/working/data"
cfg.train.ckpt_dir  = "/kaggle/working/checkpoints/kaggle_2xt4"

# --- Demo overrides (comment out for a full training run) ---
cfg.data.num_train_docs = 8000      # full config: 100000
cfg.data.num_val_docs   = 300
cfg.train.max_steps     = 2000      # full config: 40000
cfg.train.warmup_steps  = 100
cfg.train.eval_every    = 500
cfg.train.save_every    = 500
cfg.train.log_every     = 25

assert cfg.train.batch_size % jax.device_count() == 0
cfg.validate()
print("batch:", cfg.train.batch_size, "-> per-GPU:", cfg.train.batch_size // jax.device_count(),
      "| seq_len:", cfg.data.seq_len, "| dtype:", cfg.model.compute_dtype)


## 5. Prepare the data

Streams and tokenizes CodeParrot into packed memmaps under `/kaggle/working/data`.


In [ ]:
from pipeline import prepare_data
prepare_data.prepare(cfg)


## 6. Train (data-parallel over 2x T4)

Watch for the log line **`Data-parallel over 2 devices (8 examples/device).`** — that
confirms both GPUs are in use. Resume later with `train.train(cfg, resume=True)`.


In [ ]:
from pipeline import train
train.train(cfg)


## 7. Evaluate

In [ ]:
from pipeline import evaluate
evaluate.run_eval(cfg, step=None, max_batches=50)


## 8. Generate code

(Weak output after only the short demo run — train longer for good completions.)


In [ ]:
evaluate.run_generate(cfg, step=None, prompt="def quicksort(arr):\n", max_new_tokens=128)
